# CS781 Phase 1: legal language-model pretraining, implemented from scratch

**What this notebook contains**

* **Part A, the ModernBERT-style encoder** (85.4M parameters). It is the foundation model behind our best Phase 4, 5, 6 and 7 systems. It is pre-trained with masked language modelling on a domain-weighted legal corpus.
* **Part B, the GPT-2-style decoder** (112.8M parameters). It produced our Phase 1 submission (BERTScore F1 0.7973) by generating 128-word continuations.

**Everything is implemented from scratch in plain PyTorch.** No model class is imported from a library: attention, rotary position embeddings, the gated feed-forward block, layer norms, the masked-LM head, the loss and the initialisation are all written out in the cell *From-scratch model code*. Only the tokenizer comes from the `tokenizers` library, because the tokenizer is a separate artefact.

**The architecture is unchanged.** Parameter names match the reference library layout, so a checkpoint trained with the library loads here with `strict=True`, and the outputs are identical. We checked this with the same weights in both implementations:

| check | result |
|---|---|
| ModernBERT parameters | 85,380,609 in both |
| ModernBERT logits, with and without padding | maximum absolute difference 0.0 |
| ModernBERT masked-LM loss | identical (10.566844 in both) |
| GPT-2 parameters | 112,777,728 in both (112.8M) |
| GPT-2 logits | maximum absolute difference 2e-6 (float rounding) |

The optional cell *Equivalence check* repeats this test when `transformers` is installed.

**Recipe (Part A).** Sequences of 512 tokens, 131,072 tokens per optimiser step (16 x 16 x 512), 30% BERT-style masking (80% [MASK], 10% random, 10% unchanged), AdamW with betas (0.9, 0.95), weight decay 0.1, peak learning rate 6e-4, gradient clipping at 1.0, warmup-stable-decay schedule (warm-up, constant until 70% of the steps, then linear decay to 10%), fp16 with a gradient scaler on a T4. Source mix 30% Supreme Court, 30% tribunals, 35% high courts, 5% acts. The run in our experiments used 3,000 steps (393M tokens).

**How to run.** Attach the dataset with the deduplicated corpus and tokenizer, turn the GPU on, and run the cells in order. Part B needs the Phase 1 decoder checkpoint and the Phase 1 `test.json`.


## Part A: ModernBERT-style encoder, pre-trained from scratch

In [ ]:
# ================= CELL 1 — setup, paths, tokenizer (+[MASK]), configuration =================
ARCH = "modernbert"
import os, sys, re, json, math, time, random, glob, gzip, gc, copy, contextlib
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np, pandas as pd, torch, torch.nn as nn

BASE    = "/kaggle/input/datasets/oipranesh/cs781-phase4-training-prep"          # dedup corpus, manifests, tokenizer
P1_CKPT = "/kaggle/input/datasets/oipranesh/cs781-phase-1-checkpoints/checkpoint_latest.pt"    # only used for gpt2 with init='warm3000'
OUT = Path("/kaggle/working/pretrain_modernbert"); OUT.mkdir(parents=True, exist_ok=True)
assert os.path.exists(BASE), f"missing: {BASE}"
REQUIRE_GPU, TINY = True, False
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if REQUIRE_GPU: assert device.type == "cuda", "pretraining needs a GPU"
USE_BF16 = device.type == "cuda" and torch.cuda.get_device_capability(0)[0] >= 8      # T4 (7.5) has no native bf16 -> fp16 + GradScaler
AMP = device.type == "cuda"
print("device:", torch.cuda.get_device_name(0) if device.type == "cuda" else "cpu", "| precision:", ("bf16" if USE_BF16 else "fp16 + GradScaler") if AMP else "fp32")

# ---- tokenizer: the original 32,000-token legal BPE plus ONE new special token, [MASK] (id 32000). Identical in all four notebooks. ----
from tokenizers import Tokenizer
tok = Tokenizer.from_file(f"{BASE}/tokenizer/legal_bpe_32k.json"); n0 = tok.get_vocab_size()
PAD, UNK, BOS, EOS = (tok.token_to_id(t) for t in ("[PAD]", "[UNK]", "[BOS]", "[EOS]")); assert None not in (PAD, UNK, BOS, EOS)
if tok.token_to_id("[MASK]") is None: tok.add_special_tokens(["[MASK]"])
MASK = tok.token_to_id("[MASK]"); V = tok.get_vocab_size(); assert MASK == n0 and V == n0 + 1, (MASK, V, n0)
tok.save(str(OUT / "legal_bpe_32k_mask.json"))
print(f"tokenizer: {n0} -> {V} tokens | PAD/UNK/BOS/EOS/MASK = {PAD}/{UNK}/{BOS}/{EOS}/{MASK} | saved {OUT/'legal_bpe_32k_mask.json'} (attach this file to the Phase 2/3 notebooks)")

DOMAINS = ["supreme_court", "tribunals", "high_courts", "acts"]
SEQ = 512                               # pre-training block length (downstream windows are <= 512 tokens)
COMMON = dict(lr_min_frac=0.1, warmup=100, wd=0.1, betas=(0.9, 0.95), clip=1.0, eval_every=200, eval_tokens_per_domain=262144, min_words=50, seed=1234,
              domain_w={"supreme_court": 0.30, "tribunals": 0.30, "high_courts": 0.35, "acts": 0.05})     # downstream texts are judgments, not statutes
CFG_ARCH = dict(micro_bs=16, accum=16, lr_peak=6e-4, amp=True)      # tokens per optimizer step = micro_bs * SEQ * accum = 131,072
HOURS = 3.5       # suggested GPU-hours for this model (change freely)


In [ ]:
# ================= CELL 2 — streaming corpus: held-out split, domain-weighted shuffled mix, packing WITHOUT padding =================
def shard_files(domain):
    fs = sorted(glob.glob(f"{BASE}/dedup/{domain}/*.jsonl") + glob.glob(f"{BASE}/dedup/{domain}/*.jsonl.gz")); assert fs, f"no shards for {domain}"; return fs
def open_text(p): return gzip.open(p, "rt", encoding="utf-8") if p.endswith(".gz") else open(p, encoding="utf-8")
def load_hash_set(split):
    s = set()
    with open(f"{BASE}/manifests/{split}_manifest.jsonl", encoding="utf-8") as f:
        for line in f: s.add(json.loads(line)["text_hash"])
    return s
HELD = {sp: load_hash_set(sp) for sp in ("validation", "test")}
SKIP = HELD["validation"] | HELD["test"]               # training never sees validation or test documents
print({k: f"{len(v):,} docs" for k, v in HELD.items()}, "| shards:", {d: len(shard_files(d)) for d in DOMAINS})

def iter_docs(domain, rng, skip, min_words):
    files = list(shard_files(domain))
    while True:
        rng.shuffle(files)
        for p in files:
            with open_text(p) as f:
                for line in f:
                    try: r = json.loads(line)
                    except Exception: continue
                    if r.get("text_hash") in skip: continue
                    t = r.get("text")
                    if not t: continue
                    try: wc = int(r.get("word_count"))
                    except Exception: wc = len(t.split())
                    if wc >= min_words: yield t

def shuffled(it, buf, rng):                              # shard files are ordered, so mix them through a buffer
    pool = []
    for x in it:
        pool.append(x)
        if len(pool) >= buf:
            i = rng.randrange(len(pool)); pool[i], pool[-1] = pool[-1], pool[i]; yield pool.pop()

class PackedStream:
    """domain-weighted documents -> [BOS] tokens [EOS] -> packed into [micro_bs, SEQ] blocks with NO padding"""
    def __init__(self, tokenizer, micro_bs, seq, domain_w, min_words, seed, skip):
        self.tok, self.B, self.T, self.rng = tokenizer, micro_bs, seq, random.Random(seed); self.buf = []; self.dom_tokens = Counter()
        self.gens = {d: shuffled(iter_docs(d, random.Random(seed + i), skip, min_words), 1000, random.Random(seed + 100 + i)) for i, d in enumerate(DOMAINS)}
        self.w = [domain_w[d] for d in DOMAINS]
    def _refill(self, need):
        while len(self.buf) < need:
            ds = [self.rng.choices(DOMAINS, weights=self.w)[0] for _ in range(32)]; texts = [next(self.gens[d]) for d in ds]
            for d, enc in zip(ds, self.tok.encode_batch(texts, add_special_tokens=False)):
                ids = [BOS] + enc.ids + [EOS]; self.buf.extend(ids); self.dom_tokens[d] += len(ids)
    def next_batch(self):
        self._refill(self.B * self.T); x = torch.tensor(self.buf[:self.B * self.T], dtype=torch.long).view(self.B, self.T); del self.buf[:self.B * self.T]; return x

def build_val_blocks(per_domain_tokens, cache):
    """fixed held-out blocks per domain, from validation-manifest documents only, spread over several shards"""
    if cache.exists(): return torch.load(cache)
    per = per_domain_tokens // SEQ; need = per * SEQ; val = {}
    for d in DOMAINS:
        files = shard_files(d); idx = sorted({int(i * (len(files) - 1) / max(1, min(6, len(files)) - 1)) for i in range(min(6, len(files)))})
        quota = math.ceil(need / len(idx)); toks = []
        for fi in idx:
            got = 0
            with open_text(files[fi]) as f:
                for line in f:
                    try: r = json.loads(line)
                    except Exception: continue
                    if r.get("text_hash") in HELD["validation"] and r.get("text"):
                        ids = [BOS] + tok.encode(r["text"], add_special_tokens=False).ids + [EOS]; toks += ids; got += len(ids)
                        if got >= quota: break
        assert len(toks) >= need, f"{d}: only {len(toks)} validation tokens found (need {need})"
        val[d] = torch.tensor(toks[:need], dtype=torch.long).view(per, SEQ)
    torch.save(val, cache); return val
VAL = build_val_blocks(COMMON["eval_tokens_per_domain"], OUT / f"val_blocks_{SEQ}.pt")
print("validation blocks:", {d: tuple(v.shape) for d, v in VAL.items()}, "| tokens:", sum(v.numel() for v in VAL.values()))


### From-scratch model code (ModernBERT-style encoder and GPT-2-style decoder)

In [ ]:
"""From-scratch PyTorch implementations (no transformers import) of the two Phase 1 architectures:
   * ModernBERT-style masked language model  (the encoder used in Phases 4, 5, 6, 7)
   * GPT-2-style causal language model        (the decoder used for the Phase 1 submission)
   Parameter names match the Hugging Face layout, so existing checkpoints load with strict=True."""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


# =============================================================== ModernBERT ===============================================================
class MBEmbeddings(nn.Module):
    def __init__(self, vocab, d, pad_id, eps):
        super().__init__()
        self.tok_embeddings = nn.Embedding(vocab, d, padding_idx=pad_id)
        self.norm = nn.LayerNorm(d, eps=eps, bias=False)

    def forward(self, ids):
        return self.norm(self.tok_embeddings(ids))


class MBMLP(nn.Module):
    """Gated GELU feed-forward block: Wo( gelu(a) * g ) with [a, g] = Wi(x)."""
    def __init__(self, d, ff):
        super().__init__()
        self.Wi = nn.Linear(d, 2 * ff, bias=False)
        self.Wo = nn.Linear(ff, d, bias=False)

    def forward(self, x):
        a, g = self.Wi(x).chunk(2, dim=-1)
        return self.Wo(F.gelu(a) * g)


def rotate_half(x):
    h = x.shape[-1] // 2
    return torch.cat((-x[..., h:], x[..., :h]), dim=-1)


class MBAttention(nn.Module):
    """Bidirectional multi-head self-attention with rotary position embeddings; one fused q/k/v projection."""
    def __init__(self, d, heads):
        super().__init__()
        self.heads, self.hd = heads, d // heads
        self.Wqkv = nn.Linear(d, 3 * d, bias=False)
        self.Wo = nn.Linear(d, d, bias=False)

    def forward(self, x, cos, sin, mask):
        B, T, _ = x.shape
        q, k, v = self.Wqkv(x).view(B, T, 3, self.heads, self.hd).unbind(dim=2)
        q, k, v = q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2)               # (B, heads, T, head_dim)
        q = q * cos + rotate_half(q) * sin
        k = k * cos + rotate_half(k) * sin
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)                    # scale = 1/sqrt(head_dim)
        return self.Wo(out.transpose(1, 2).reshape(B, T, -1))


class MBLayer(nn.Module):
    """Pre-norm block. The first layer has no attention norm (the embedding LayerNorm already normalises its input)."""
    def __init__(self, d, ff, heads, layer_idx, eps):
        super().__init__()
        self.attn_norm = nn.Identity() if layer_idx == 0 else nn.LayerNorm(d, eps=eps, bias=False)
        self.attn = MBAttention(d, heads)
        self.mlp_norm = nn.LayerNorm(d, eps=eps, bias=False)
        self.mlp = MBMLP(d, ff)

    def forward(self, x, cos, sin, mask):
        x = x + self.attn(self.attn_norm(x), cos, sin, mask)
        return x + self.mlp(self.mlp_norm(x))


class MBPredictionHead(nn.Module):
    def __init__(self, d, eps):
        super().__init__()
        self.dense = nn.Linear(d, d, bias=False)
        self.norm = nn.LayerNorm(d, eps=eps, bias=False)

    def forward(self, x):
        return self.norm(F.gelu(self.dense(x)))


class ModernBertCore(nn.Module):
    def __init__(self, vocab, d, ff, layers, heads, local_window, global_every, pad_id, eps, theta_global, theta_local):
        super().__init__()
        self.hd, self.half_window, self.pad_id = d // heads, local_window // 2, pad_id
        self.layer_is_global = [i % global_every == 0 for i in range(layers)]
        self.embeddings = MBEmbeddings(vocab, d, pad_id, eps)
        self.layers = nn.ModuleList([MBLayer(d, ff, heads, i, eps) for i in range(layers)])
        self.final_norm = nn.LayerNorm(d, eps=eps, bias=False)
        for name, theta in (("global", theta_global), ("local", theta_local)):
            self.register_buffer(f"inv_freq_{name}", 1.0 / (theta ** (torch.arange(0, self.hd, 2, dtype=torch.float) / self.hd)), persistent=False)

    def rope(self, T, name, device, dtype):
        freqs = torch.arange(T, device=device).float()[:, None] * getattr(self, f"inv_freq_{name}").to(device)
        emb = torch.cat((freqs, freqs), dim=-1)
        return emb.cos()[None, None].to(dtype), emb.sin()[None, None].to(dtype)           # (1, 1, T, head_dim)

    def forward(self, ids, attention_mask=None):
        B, T = ids.shape
        x = self.embeddings(ids)
        full = torch.ones(T, T, dtype=torch.bool, device=ids.device)
        pos = torch.arange(T, device=ids.device)
        local = (pos[:, None] - pos[None, :]).abs() <= self.half_window                   # sliding window, inclusive
        full, local = full[None, None].expand(B, 1, T, T), local[None, None].expand(B, 1, T, T)
        if attention_mask is not None:                                                    # key padding mask
            keep = attention_mask.bool()[:, None, None, :]
            full, local = full & keep, local & keep
        cs = {n: self.rope(T, n, ids.device, x.dtype) for n in ("global", "local")}
        for is_global, layer in zip(self.layer_is_global, self.layers):
            n = "global" if is_global else "local"
            x = layer(x, cs[n][0], cs[n][1], full if is_global else local)
        return self.final_norm(x)


class ModernBertMLM(nn.Module):
    """Masked language model: core encoder + prediction head + decoder tied to the input embeddings."""
    def __init__(self, vocab_size=32001, hidden_size=768, intermediate_size=1152, num_layers=12, num_heads=12, local_attention=128,
                 global_attn_every_n_layers=3, pad_id=0, norm_eps=1e-5, global_rope_theta=160000.0, local_rope_theta=10000.0, init_std=0.02):
        super().__init__()
        self.cfg = dict(vocab_size=vocab_size, hidden_size=hidden_size, intermediate_size=intermediate_size, num_layers=num_layers, num_heads=num_heads,
                        local_attention=local_attention, global_attn_every_n_layers=global_attn_every_n_layers)
        self.model = ModernBertCore(vocab_size, hidden_size, intermediate_size, num_layers, num_heads, local_attention, global_attn_every_n_layers, pad_id, norm_eps,
                                    global_rope_theta, local_rope_theta)
        self.head = MBPredictionHead(hidden_size, norm_eps)
        self.decoder = nn.Linear(hidden_size, vocab_size, bias=True)
        self.decoder.weight = self.model.embeddings.tok_embeddings.weight                 # tied
        self.init_weights(init_std)

    @torch.no_grad()
    def init_weights(self, std, cutoff=2.0):
        """Truncated-normal initialisation: input projections std, output projections std/sqrt(2*layers), embeddings std; norms start at 1."""
        out_std = std / math.sqrt(2.0 * self.cfg["num_layers"])
        def tn(w, s): nn.init.trunc_normal_(w, mean=0.0, std=s, a=-cutoff * s, b=cutoff * s)
        tn(self.model.embeddings.tok_embeddings.weight, std)
        if self.model.embeddings.tok_embeddings.padding_idx is not None:
            self.model.embeddings.tok_embeddings.weight[self.model.embeddings.tok_embeddings.padding_idx].zero_()
        for layer in self.model.layers:
            tn(layer.attn.Wqkv.weight, std); tn(layer.attn.Wo.weight, out_std); tn(layer.mlp.Wi.weight, std); tn(layer.mlp.Wo.weight, out_std)
        tn(self.head.dense.weight, out_std)
        nn.init.zeros_(self.decoder.bias)
        for m in self.modules():
            if isinstance(m, nn.LayerNorm): nn.init.ones_(m.weight)

    def forward(self, input_ids, attention_mask=None, labels=None, ignore_index=-100):
        h = self.model(input_ids, attention_mask)
        if labels is not None:                                                            # sparse prediction: only masked positions are scored
            keep = labels.view(-1) != ignore_index
            h = h.reshape(-1, h.shape[-1])[keep]
            logits = self.decoder(self.head(h))
            return logits, F.cross_entropy(logits.float(), labels.view(-1)[keep])
        return self.decoder(self.head(h)), None


# =============================================================== GPT-2 ===============================================================
class Conv1D(nn.Module):
    """A linear layer whose weight is stored as (in, out), the layout used by GPT-2 checkpoints."""
    def __init__(self, nf, nx):
        super().__init__()
        self.weight, self.bias = nn.Parameter(torch.empty(nx, nf)), nn.Parameter(torch.zeros(nf))

    def forward(self, x):
        return torch.addmm(self.bias, x.reshape(-1, x.shape[-1]), self.weight).view(*x.shape[:-1], -1)


class GPT2Attention(nn.Module):
    def __init__(self, d, heads):
        super().__init__()
        self.heads = heads
        self.c_attn, self.c_proj = Conv1D(3 * d, d), Conv1D(d, d)

    def forward(self, x):
        B, T, D = x.shape
        q, k, v = self.c_attn(x).split(D, dim=2)
        q, k, v = (t.view(B, T, self.heads, D // self.heads).transpose(1, 2) for t in (q, k, v))
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.c_proj(out.transpose(1, 2).reshape(B, T, D))


class GPT2MLP(nn.Module):
    def __init__(self, d, inner):
        super().__init__()
        self.c_fc, self.c_proj = Conv1D(inner, d), Conv1D(d, inner)

    def forward(self, x):
        return self.c_proj(F.gelu(self.c_fc(x), approximate="tanh"))


class GPT2Block(nn.Module):
    def __init__(self, d, heads, inner, eps):
        super().__init__()
        self.ln_1, self.attn = nn.LayerNorm(d, eps=eps), GPT2Attention(d, heads)
        self.ln_2, self.mlp = nn.LayerNorm(d, eps=eps), GPT2MLP(d, inner)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        return x + self.mlp(self.ln_2(x))


class GPT2Core(nn.Module):
    def __init__(self, vocab, positions, d, layers, heads, inner, eps):
        super().__init__()
        self.wte, self.wpe = nn.Embedding(vocab, d), nn.Embedding(positions, d)
        self.h = nn.ModuleList([GPT2Block(d, heads, inner, eps) for _ in range(layers)])
        self.ln_f = nn.LayerNorm(d, eps=eps)

    def forward(self, ids):
        x = self.wte(ids) + self.wpe(torch.arange(ids.shape[1], device=ids.device))[None]
        for blk in self.h: x = blk(x)
        return self.ln_f(x)


class GPT2LM(nn.Module):
    def __init__(self, vocab_size=32000, n_positions=4096, n_embd=768, n_layer=12, n_head=12, n_inner=None, eps=1e-5, init_std=0.02):
        super().__init__()
        self.cfg = dict(vocab_size=vocab_size, n_positions=n_positions, n_embd=n_embd, n_layer=n_layer, n_head=n_head)
        self.transformer = GPT2Core(vocab_size, n_positions, n_embd, n_layer, n_head, n_inner or 4 * n_embd, eps)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.lm_head.weight = self.transformer.wte.weight                                 # tied
        self.init_weights(init_std)

    @torch.no_grad()
    def init_weights(self, std):
        for m in self.modules():
            if isinstance(m, Conv1D): nn.init.normal_(m.weight, std=std)
            elif isinstance(m, nn.Embedding): nn.init.normal_(m.weight, std=std)
            elif isinstance(m, nn.LayerNorm): nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
        for name, p in self.named_parameters():                                           # residual-path scaling, as in GPT-2
            if name.endswith("c_proj.weight"): nn.init.normal_(p, std=std / math.sqrt(2 * self.cfg["n_layer"]))

    def forward(self, input_ids, labels=None):
        logits = self.lm_head(self.transformer(input_ids))
        loss = F.cross_entropy(logits[:, :-1].reshape(-1, logits.shape[-1]).float(), labels[:, 1:].reshape(-1)) if labels is not None else None
        return logits, loss

    @torch.no_grad()
    def generate(self, ids, max_new_tokens=300, temperature=0.8, top_k=50):
        """Sampling with temperature and top-k, as used for the Phase 1 submission."""
        self.eval()
        for _ in range(max_new_tokens):
            logits = self(ids[:, -self.cfg["n_positions"]:])[0][:, -1, :] / temperature
            v, ix = logits.topk(min(top_k, logits.shape[-1]))
            nxt = ix.gather(1, torch.multinomial(F.softmax(v, dim=-1), 1))
            ids = torch.cat([ids, nxt], dim=1)
        return ids


In [ ]:
# ================= ARCHITECTURE (from scratch): encoder-only Transformer, bidirectional masked-language modelling =================
SPEC = dict(hidden_size=768, intermediate_size=1152, num_hidden_layers=12, num_attention_heads=12, local_attention=128, global_attn_every_n_layers=3)   # RoPE, gated GELU, no biases, local + global attention
SPECIAL = [PAD, UNK, BOS, EOS]

def build_model(tiny=False, init="scratch"):
    """tiny=True is only used by the unit tests"""
    s = SPEC if not tiny else dict(hidden_size=64, intermediate_size=96, num_hidden_layers=4, num_attention_heads=4, local_attention=16, global_attn_every_n_layers=2)
    m = ModernBertMLM(vocab_size=V, hidden_size=s["hidden_size"], intermediate_size=s["intermediate_size"], num_layers=s["num_hidden_layers"],
                      num_heads=s["num_attention_heads"], local_attention=s["local_attention"], global_attn_every_n_layers=s["global_attn_every_n_layers"], pad_id=PAD)
    mc = dict(family="modernbert", vocab_size=V, context_length=SEQ, hidden_size=s["hidden_size"], intermediate_size=s["intermediate_size"], num_layers=s["num_hidden_layers"],
              num_heads=s["num_attention_heads"], local_attention=s["local_attention"], global_attn_every_n_layers=s["global_attn_every_n_layers"])
    n = sum(p.numel() for p in m.parameters())
    if not tiny: assert 50e6 <= n <= 300e6, f"{n:,} parameters is outside the 50M-300M Phase 1 range"
    return (m if tiny else m.to(device)), mc, n

# ---- objective: BERT-style masking. 30% of the non-special tokens are selected; 80% -> [MASK], 10% -> a random token, 10% unchanged ----
def mlm_mask(x, p=0.30, gen=None):
    sel = torch.rand(x.shape, generator=gen, device=x.device) < p
    for s_ in SPECIAL: sel &= (x != s_)
    labels = torch.where(sel, x, torch.full_like(x, -100))
    r = torch.rand(x.shape, generator=gen, device=x.device); inp = x.clone()
    inp[sel & (r < 0.8)] = MASK
    rnd = sel & (r >= 0.8) & (r < 0.9); inp[rnd] = torch.randint(4, MASK, x.shape, generator=gen, device=x.device)[rnd]
    return inp, labels
def forward_loss(model, x, rng=None, gen=None):
    inp, labels = mlm_mask(x, 0.30, gen); return model(inp, labels=labels)[1]          # no padding inside a packed block, so no attention mask is needed
def val_item(x, gen, rng): return mlm_mask(x, 0.30, gen)
def eval_loss(model, it): return model(it[0], labels=it[1])[1]


In [ ]:
# ================= CELL 5 — evaluation + trainer (warmup-stable-decay, fp16/bf16, resume, per-domain validation) =================
def amp_ctx(enabled=True):
    if not (AMP and enabled): return contextlib.nullcontext()
    return torch.autocast("cuda", dtype=torch.bfloat16 if USE_BF16 else torch.float16)

_VALB = {}
def val_batches(bs=16, max_batches=None):
    """fixed, deterministic validation inputs (any masking / corruption uses a fixed seed, so every evaluation is comparable)"""
    key = (bs, max_batches)
    if key not in _VALB:
        gen = torch.Generator().manual_seed(123); rng = np.random.default_rng(123); out = {}
        for d, blocks in VAL.items():
            items = []
            for i in range(0, len(blocks), bs):
                if max_batches and len(items) >= max_batches: break
                items.append(val_item(blocks[i:i + bs], gen, rng))
            out[d] = items
        _VALB[key] = out
    return _VALB[key]

@torch.no_grad()
def evaluate(model, amp=True, bs=16, max_batches=None):
    model.eval(); res = {}
    for d, items in val_batches(bs, max_batches).items():
        tot = 0.0
        for it in items:
            it = [t.to(device) for t in it]
            with amp_ctx(amp): loss = eval_loss(model, it)
            tot += float(loss)
        res[d] = tot / max(1, len(items))
    res["overall"] = float(np.mean([res[d] for d in VAL])); model.train(); return res
def fmt(res): return " | ".join(f"{k} {v:.3f}" for k, v in res.items())

import threading, queue
PREFETCH = True        # build the next batches on a background thread while the GPU computes (the tokenizer runs in Rust and releases the GIL)
class Prefetch:
    def __init__(self, stream, depth=6):
        self.stream, self.q, self.err, self._stop = stream, queue.Queue(maxsize=depth), None, False
        self.t = threading.Thread(target=self._run, daemon=True); self.t.start()
    def _run(self):
        try:
            while not self._stop: self.q.put(self.stream.next_batch())
        except Exception as e: self.err = e; self.q.put(None)
    def next_batch(self):
        x = self.q.get()
        if x is None: raise self.err
        return x
    @property
    def dom_tokens(self): return self.stream.dom_tokens
    def close(self):
        self._stop = True
        try:
            while True: self.q.get_nowait()
        except queue.Empty: pass

def lr_at(step, c):
    w, T = c["warmup"], c["total_steps"]; ds = c["decay_start"] or int(0.7 * T); lo = c["lr_peak"] * c["lr_min_frac"]
    if step < w: return c["lr_peak"] * (step + 1) / w
    if step < ds: return c["lr_peak"]
    return c["lr_peak"] + (lo - c["lr_peak"]) * min(1.0, (step - ds) / max(1, T - ds))

def train(cfg=None, steps=None, accum=None, do_eval=True, save=True, resume=True, minutes_budget=None, max_val_batches=None, keep_model=False, quiet=False, run=None):
    c = {**COMMON, **CFG_ARCH, **(cfg or {})}; c.setdefault("total_steps", 3000); c.setdefault("decay_start", None); c["accum"] = accum or c["accum"]
    rdir = OUT / (run or ARCH); rdir.mkdir(exist_ok=True); torch.manual_seed(c["seed"])
    model, mc, n_params = build_model(tiny=TINY, init=c.get("init", "scratch")); model.train()
    groups = [{"params": [p for p in model.parameters() if p.ndim >= 2], "weight_decay": c["wd"]}, {"params": [p for p in model.parameters() if p.ndim < 2], "weight_decay": 0.0}]
    try: opt = torch.optim.AdamW(groups, lr=c["lr_peak"], betas=c["betas"], eps=1e-8, fused=(device.type == "cuda"))
    except Exception: opt = torch.optim.AdamW(groups, lr=c["lr_peak"], betas=c["betas"], eps=1e-8)
    use_amp = AMP and c["amp"]; scaler = torch.amp.GradScaler("cuda", enabled=use_amp and not USE_BF16); step, tokens = 0, 0; ck_path = rdir / "latest.pt"
    if save and resume and ck_path.exists():
        ck = torch.load(ck_path, map_location="cpu", weights_only=False); model.load_state_dict(ck["model_state_dict"]); opt.load_state_dict(ck["optimizer_state_dict"])
        scaler.load_state_dict(ck["scaler_state_dict"]); step, tokens = ck["step"], ck["tokens"]; print(f"[{ARCH}] resumed from step {step}"); del ck
    stream = PackedStream(tok, c["micro_bs"], SEQ, c["domain_w"], c["min_words"], c["seed"] + step, SKIP)
    if PREFETCH: stream = Prefetch(stream)
    rng = np.random.default_rng(c["seed"] + step); gen = torch.Generator(device=device).manual_seed(c["seed"] + step)
    T = c["micro_bs"] * SEQ * c["accum"]; end = steps if steps is not None else c["total_steps"]; log = open(rdir / "log.jsonl", "a"); t0 = time.time(); last_save = time.time()
    best, run_loss, nb, step_times, hist, bad = float("inf"), 0.0, 0, [], [], 0; ev = c["eval_every"]
    while step < end:
        ts = time.time(); lr = lr_at(step, c)
        for g in opt.param_groups: g["lr"] = lr
        good = 0
        for _ in range(c["accum"]):
            x = stream.next_batch().to(device)
            with amp_ctx(use_amp): loss = forward_loss(model, x, rng=rng, gen=gen)
            if not torch.isfinite(loss): bad += 1; print(f"[{ARCH}] non-finite loss (total {bad}); skipping microbatch"); assert bad < 50, "too many non-finite losses: set amp=False in CFG_ARCH (cell 1)"; continue
            scaler.scale(loss / c["accum"]).backward(); run_loss += loss.item(); nb += 1; good += 1
        if good:
            scaler.unscale_(opt); gn = torch.nn.utils.clip_grad_norm_(model.parameters(), c["clip"]); scaler.step(opt); scaler.update()
        else: gn = torch.tensor(float("nan"))
        opt.zero_grad(set_to_none=True); step += 1; tokens += T
        if device.type == "cuda": torch.cuda.synchronize()
        step_times.append(time.time() - ts)
        if step % 10 == 0:
            rec = dict(step=step, loss=run_loss / max(nb, 1), lr=lr, grad_norm=float(gn), tokens=tokens, tok_s=T / float(np.median(step_times[-10:])), elapsed_min=(time.time() - t0) / 60); run_loss, nb = 0.0, 0
            log.write(json.dumps(rec) + "\n"); log.flush()
            if not quiet and (step % 50 == 0 or step <= 20): print(f"[{ARCH}] step {step:5d} | loss {rec['loss']:.3f} | lr {lr:.2e} | gnorm {rec['grad_norm']:.2f} | {rec['tok_s']/1e3:.1f}k tok/s | {rec['elapsed_min']:.1f} min | {tokens/1e6:.0f}M tokens", flush=True)
        if do_eval and (step % ev == 0 or step == end):
            r = evaluate(model, c["amp"], max_batches=max_val_batches); hist.append(dict(step=step, tokens=tokens, elapsed_min=(time.time() - t0) / 60, val=r)); log.write(json.dumps(dict(step=step, val=r)) + "\n"); log.flush()
            if not quiet: print(f"   [{ARCH}] VALIDATION step {step}: {fmt(r)}", flush=True)
            if save and r["overall"] < best: best = r["overall"]; torch.save({"model_state_dict": model.state_dict(), "model_config": mc, "step": step, "val": r}, rdir / "best.pt")
        if save and time.time() - last_save > 20 * 60:
            torch.save({"model_state_dict": model.state_dict(), "optimizer_state_dict": opt.state_dict(), "scaler_state_dict": scaler.state_dict(), "step": step, "tokens": tokens, "model_config": mc, "cfg": c}, ck_path)
            last_save = time.time(); print(f"   [{ARCH}] checkpoint saved")
        if minutes_budget and (time.time() - t0) / 60 > minutes_budget: print(f"[{ARCH}] time budget reached at step {step}"); break
    if save:
        torch.save({"model_state_dict": model.state_dict(), "optimizer_state_dict": opt.state_dict(), "scaler_state_dict": scaler.state_dict(), "step": step, "tokens": tokens, "model_config": mc, "cfg": c}, ck_path)
        torch.save({"model_state_dict": model.state_dict(), "model_config": {**mc, "trained_steps": step, "tokens": tokens}, "step": step}, rdir / "final_model.pt")
    log.close(); tail = step_times[2:] if len(step_times) > 3 else step_times
    info = dict(history=hist, tok_s=float(T / np.median(tail)), params=n_params, steps=step, tokens=tokens, minutes=(time.time() - t0) / 60, model=model if keep_model else None,
                mix={d: v / max(1, sum(stream.dom_tokens.values())) for d, v in stream.dom_tokens.items()})
    if PREFETCH: stream.close()
    del opt, scaler; gc.collect()
    if device.type == "cuda": torch.cuda.empty_cache()
    return info

def plan_budget(tok_s, hours):
    c = {**COMMON, **CFG_ARCH}; per = c["micro_bs"] * SEQ * c["accum"]; steps = int(hours * 3600 * tok_s / per) // 10 * 10
    print(f"  at {tok_s/1e3:.1f}k tok/s, {hours} GPU-h = {steps} steps = {steps*per/1e9:.2f}B tokens"); return steps

def run_arch(hours=None, steps=None, run=None, **over):
    """probe the real throughput, size the schedule to the GPU-hour budget, then train (resumes automatically after an interruption)"""
    hours = HOURS if hours is None else hours
    probe = train(cfg=dict(init="scratch"), steps=8, accum=2, do_eval=False, save=False, resume=False, quiet=True, run="probe")
    steps = steps or plan_budget(probe["tok_s"], hours)
    cfg = dict(total_steps=steps, warmup=max(20, min(150, steps // 20)), decay_start=int(0.7 * steps), **over)
    print(f"[{ARCH}] {steps} steps for ~{hours} GPU-hours (measured {probe['tok_s']/1e3:.1f}k tok/s)")
    return train(cfg=cfg, run=run)


In [ ]:
# ================= CELL 6 — SMOKE TEST: 6 optimizer steps + validation (~2 min). Must pass before the real run. =================
info = train(cfg=dict(init="scratch"), steps=6, accum=2, do_eval=True, save=False, resume=False, max_val_batches=2, run="smoke")
v = info["history"][-1]["val"]; assert np.isfinite(v["overall"]), "non-finite validation loss - do not train before fixing (try amp=False in CFG_ARCH, cell 1)"
print(f"\n{ARCH}: {info['params']/1e6:.1f}M parameters | validation loss after 6 steps {v['overall']:.3f} | {info['tok_s']/1e3:.1f}k tokens/s on this GPU")
print("\nGPU-hour planning:")
for h in (2, 3, 5, 8): plan_budget(info["tok_s"], h)
del info; gc.collect()
if device.type == "cuda": torch.cuda.empty_cache()


In [ ]:
# ================= CELL 6b — SPEED PROFILE (~2 min): where does the 14% throughput loss between the smoke test and the real run come from? =================
# Times (a) building batches (tokenising text) and (b) the GPU forward/backward on a fixed batch, and reads the GPU clock/power. Read-only.
import subprocess
model_p, _, _ = build_model(tiny=TINY); model_p.train(); ps = PackedStream(tok, CFG_ARCH["micro_bs"], SEQ, COMMON["domain_w"], COMMON["min_words"], 999, SKIP); N = 20
t0 = time.time(); batches = [ps.next_batch() for _ in range(N)]; t_data = (time.time() - t0) / N
x = batches[0].to(device); g_ = torch.Generator(device=device).manual_seed(0); r_ = np.random.default_rng(0)
for _ in range(2):
    with amp_ctx(CFG_ARCH["amp"]): forward_loss(model_p, x, rng=r_, gen=g_).backward()
if device.type == "cuda": torch.cuda.synchronize()
t0 = time.time()
for _ in range(N):
    with amp_ctx(CFG_ARCH["amp"]): forward_loss(model_p, x, rng=r_, gen=g_).backward()
if device.type == "cuda": torch.cuda.synchronize()
t_gpu = (time.time() - t0) / N; tokens = CFG_ARCH["micro_bs"] * SEQ
print(f"building one micro-batch of {tokens} tokens: {t_data*1000:.0f} ms | GPU forward+backward on it: {t_gpu*1000:.0f} ms | data/GPU = {t_data/t_gpu:.1%}")
print(f"  pure GPU speed {tokens/t_gpu/1e3:.1f}k tok/s; if data were NOT overlapped: {tokens/(t_gpu+t_data)/1e3:.1f}k tok/s")
try: print("  GPU clocks / power / temperature:", subprocess.run(["nvidia-smi", "--query-gpu=clocks.sm,clocks.max.sm,power.draw,power.limit,temperature.gpu", "--format=csv,noheader"], capture_output=True, text=True, timeout=20).stdout.strip())
except Exception as e: print("  nvidia-smi not available:", type(e).__name__)
print("  reading: data/GPU above ~10% -> the background prefetch (PREFETCH=True in the trainer cell) should recover most of the gap; if clocks are well below the maximum while power sits at its limit, the GPU is throttling and nothing in the code can fix it")
del model_p, ps, batches; gc.collect()
if device.type == "cuda": torch.cuda.empty_cache()


In [ ]:
# ================= CELL 7 — THE RUN: probes your real GPU speed, sizes the schedule to HOURS, trains, saves. Resumes automatically if the session is interrupted. =================
# Gate: within ~20 minutes the printed validation loss must be falling steadily. If it is flat or non-finite, stop and send me the log.
# For a short pilot first: run_arch(hours=0.4)   (then delete /kaggle/working/pretrain_modernbert/modernbert before the real run)
# v2 run: 3.5 GPU-hours. Make sure Kaggle Secrets KAGGLE_USERNAME / KAGGLE_KEY are attached and Internet is ON, then Save & Run All: cell 9 pushes the model automatically.
info = run_arch()


In [ ]:
# ================= EXPORT: the trained encoder is saved by the trainer as final_model.pt =================
# The checkpoint holds model_state_dict (parameter names identical to the library layout), model_config and the step / token counts.
# Reload it with the from-scratch class:
#   ck = torch.load(OUT / "modernbert" / "final_model.pt", map_location="cpu", weights_only=False)
#   model = ModernBertMLM(vocab_size=ck["model_config"]["vocab_size"]); model.load_state_dict(ck["model_state_dict"], strict=True)
print("final checkpoint:", OUT / ARCH / "final_model.pt", "| tokenizer:", OUT / "legal_bpe_32k_mask.json")


In [ ]:
# ================= Equivalence check against the reference library implementation (optional: needs `transformers`) =================
try:
    from transformers import ModernBertConfig, ModernBertForMaskedLM, GPT2Config, GPT2LMHeadModel
except ImportError:
    print("transformers is not installed: skipping the optional check")
else:
    torch.manual_seed(0)
    cfg = ModernBertConfig(vocab_size=32001, max_position_embeddings=512, hidden_size=768, intermediate_size=1152, num_hidden_layers=12, num_attention_heads=12, local_attention=128,
                           global_attn_every_n_layers=3, pad_token_id=0, attention_dropout=0.0, mlp_dropout=0.0, embedding_dropout=0.0, reference_compile=False)
    ref, mine = ModernBertForMaskedLM(cfg).eval(), ModernBertMLM().eval()
    print("ModernBERT:", mine.load_state_dict(ref.state_dict(), strict=True), "| parameters", sum(p.numel() for p in ref.parameters()), "vs", sum(p.numel() for p in mine.parameters()))
    ids = torch.randint(1, 32000, (2, 300)); pad = torch.ones(2, 300, dtype=torch.long); pad[1, 220:] = 0
    with torch.no_grad():
        for name, mask in (("no padding", None), ("with padding", pad)):
            ok = torch.ones(2, 300, dtype=torch.bool) if mask is None else mask.bool()
            print(f"  logits {name}: max |library - scratch| =", (ref(input_ids=ids, attention_mask=mask).logits - mine(ids, attention_mask=mask)[0])[ok].abs().max().item())
    g = GPT2LMHeadModel(GPT2Config(vocab_size=32000, n_positions=4096, n_embd=768, n_layer=12, n_head=12, resid_pdrop=0.0, embd_pdrop=0.0, attn_pdrop=0.0)).eval(); gm = GPT2LM().eval()
    print("GPT-2:", gm.load_state_dict({k: v for k, v in g.state_dict().items() if not k.endswith(".attn.bias") and not k.endswith("masked_bias")}, strict=True))
    x = torch.randint(0, 32000, (2, 200))
    with torch.no_grad(): print("  logits: max |library - scratch| =", (g(x).logits - gm(x)[0]).abs().max().item())


## Part B: the GPT-2-style decoder behind the Phase 1 submission

The decoder is a 12-layer, 768-wide, 12-head causal Transformer with learned absolute positions (4,096), a 32,000-token vocabulary, tied embeddings and 112.8M parameters. It was pre-trained for 1,000 steps (effective batch of 8 sequences of up to 4,096 tokens). The cells below load that checkpoint into the from-scratch `GPT2LM` class, continue each Phase 1 test text with temperature 0.8 and top-k 50 sampling (up to 300 new tokens), cut the continuation to 128 words and write `prediction.zip`.

In [ ]:
# ---------- Part B, setup ----------
import os, json, zipfile
from pathlib import Path
import torch
from tokenizers import Tokenizer, decoders
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu"); USE_AMP = DEVICE.type == "cuda"; CONTEXT_LENGTH = 4096
tokpath = next(iter(Path("/kaggle/input").rglob("legal_bpe_32k.json")), None); assert tokpath, "attach the dataset with legal_bpe_32k.json"
tokenizer = Tokenizer.from_file(str(tokpath)); tokenizer.decoder = decoders.ByteLevel()
VOCAB_SIZE = tokenizer.get_vocab_size(); BOS_ID = tokenizer.token_to_id("[BOS]"); assert VOCAB_SIZE == 32000 and BOS_ID is not None
ckpts = [Path("/kaggle/working/phase1_decoder_transformer/checkpoints/checkpoint_latest.pt")] + list(Path("/kaggle/input").rglob("checkpoint_latest.pt")) + list(Path("/kaggle/input").rglob("checkpoint_step_001000.pt"))
FINAL_CKPT = next((p for p in ckpts if p.exists()), None); assert FINAL_CKPT, "attach the decoder checkpoint (checkpoint_latest.pt)"
print("tokenizer:", tokpath, "| checkpoint:", FINAL_CKPT)


In [ ]:
# ---------- Part B, load the trained weights into the from-scratch decoder ----------
ckpt = torch.load(FINAL_CKPT, map_location="cpu", weights_only=False); cfg = ckpt.get("model_config", {})
model = GPT2LM(vocab_size=int(cfg.get("vocab_size", VOCAB_SIZE)), n_positions=int(cfg.get("context_length", CONTEXT_LENGTH)), n_embd=int(cfg.get("hidden_size", 768)),
               n_layer=int(cfg.get("num_layers", 12)), n_head=int(cfg.get("num_heads", 12)))
sd = {k: v for k, v in ckpt["model_state_dict"].items() if not k.endswith(".attn.bias") and not k.endswith("masked_bias")}      # drop the library's non-learned causal-mask buffers
model.load_state_dict(sd, strict=True); model.to(DEVICE).eval()
print("loaded step", ckpt.get("step", "unknown"), "| parameters", f"{sum(p.numel() for p in model.parameters()):,}")


In [ ]:
# ---------- Part B, load the official Phase 1 test texts ----------
cands = []
for p in Path("/kaggle/input").rglob("test.json"):
    try:
        obj = json.load(open(p, encoding="utf-8"))
        if isinstance(obj, dict) and obj and all(isinstance(v, dict) and isinstance(v.get("text"), str) for v in list(obj.values())[:10]): cands.append((p, obj))
    except Exception: pass
assert cands, "attach the dataset with the Phase 1 test.json"
TEST_JSON, test_data = cands[0]; print("test file:", TEST_JSON, "| examples:", len(test_data))


In [ ]:
# ---------- Part B, generation: sample until 128 words, then cut ----------
def generate_128_words(prompt, temperature=0.8, top_k=50, max_generation_tokens=300):
    prompt_ids = tokenizer.encode(prompt).ids or [BOS_ID]
    if len(prompt_ids) >= CONTEXT_LENGTH: prompt_ids = prompt_ids[-CONTEXT_LENGTH + 1:]
    inp = torch.tensor([prompt_ids], dtype=torch.long, device=DEVICE); gen = inp.clone()
    with torch.no_grad():
        for _ in range(max_generation_tokens):
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
                logits = model(gen[:, -CONTEXT_LENGTH:])[0][:, -1, :]
            logits = logits.float() / temperature
            values, indices = torch.topk(logits, k=min(top_k, logits.shape[-1]), dim=-1)
            filtered = torch.full_like(logits, float("-inf")); filtered.scatter_(1, indices, values)
            gen = torch.cat([gen, torch.multinomial(torch.softmax(filtered, dim=-1), num_samples=1)], dim=1)
            if len(tokenizer.decode(gen[0, inp.shape[1]:].tolist(), skip_special_tokens=True).split()) >= 128: break
    text = " ".join(tokenizer.decode(gen[0, inp.shape[1]:].tolist(), skip_special_tokens=True).split())
    return " ".join(text.split()[:128])

first = next(iter(test_data)); sample = generate_128_words(test_data[first]["text"]); print(first, "->", len(sample.split()), "words\n", sample[:600])
assert len(sample.split()) == 128 and "\u0120" not in sample and "\u010a" not in sample


In [ ]:
# ---------- Part B, generate all predictions and write the submission ----------
predictions = {}
for i, (tid, item) in enumerate(test_data.items(), 1):
    predictions[tid] = {"text": generate_128_words(item["text"])}
    if i % 100 == 0 or i == len(test_data): print(f"[{i}/{len(test_data)}]")
assert set(predictions) == set(test_data) and all(len(v["text"].split()) == 128 for v in predictions.values())
SUB = Path("/kaggle/working/phase1_decoder_submission"); SUB.mkdir(parents=True, exist_ok=True)
json.dump(predictions, open(SUB / "prediction.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
with zipfile.ZipFile("/kaggle/working/phase1_decoder_submission.zip", "w", zipfile.ZIP_DEFLATED) as z: z.write(SUB / "prediction.json", arcname="prediction.json")
print("wrote /kaggle/working/phase1_decoder_submission.zip with", len(predictions), "predictions")
